# LLM Explainability Layer: tuned XGBoost + SHAP + ChatGPT

Ноутбук собирает explainability-слой поверх оттюненного XGBoost-регрессора (`duration_hours`):

1. Обучаем XGB на тех же параметрах, что в `artifacts_ml_tuning/final_summary.json` (model = `XGB_reg`).
2. Считаем SHAP через `TreeExplainer` — настоящие значения модели, без прокси.
3. LLM-слой (ChatGPT) получает SHAP top-факторы + метаданные задачи и объясняет:
   - почему модель дала такой прогноз,
   - где узкие места (что сильнее всего увеличивает срок),
   - что сделать для сокращения.

Требования: `xgboost`, `shap`, `openai>=1.0`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`.
Ключ OpenAI берётся из переменной окружения `OPENAI_API_KEY` (или задаётся в ячейке ниже).

In [ ]:
os.environ['OPENAI_API_KEY'] = ''


## 1. Импорты и пути

In [ ]:
import json, os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

import xgboost as xgb
import shap

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data_finall_without_TTM.csv')
TUNING_SUMMARY = os.path.join(BASE_DIR, 'artifacts_ml_tuning', 'final_summary.json')

TARGET = 'duration_hours'
DURATION_CAP = 960
SEED = 42

print('xgboost', xgb.__version__, '| shap', shap.__version__)

## 2. Данные и split (как в `export_models.py` / `ML_tuning.ipynb`)

In [ ]:
df = pd.read_csv(DATA_PATH)
print('rows:', len(df), '| cols:', df.shape[1])

feature_cols = [c for c in df.columns if c != TARGET]

bool_cols = df[feature_cols].select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    df[bool_cols] = df[bool_cols].astype(np.uint8)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full  = df.iloc[split:].copy()

train = train_full[train_full[TARGET] < DURATION_CAP].copy()
test  = test_full[test_full[TARGET]  < DURATION_CAP].copy()

X_train = train[feature_cols].values.astype(np.float32)
y_train = train[TARGET].values.astype(np.float32)
X_test  = test[feature_cols].values.astype(np.float32)
y_test  = test[TARGET].values.astype(np.float32)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)
np.nan_to_num(X_train_s, copy=False)
np.nan_to_num(X_test_s,  copy=False)

print('train:', X_train_s.shape, '| test:', X_test_s.shape)

## 3. Подгружаем гиперпараметры оттюненного XGB

In [ ]:
with open(TUNING_SUMMARY, encoding='utf-8') as f:
    summary = json.load(f)

xgb_entry = next(m for m in summary if m['model'] == 'XGB_reg')
print('Selected from:', xgb_entry['selected_from'])
print('cv_MAE:', xgb_entry['cv_MAE'], '| holdout MAE:', xgb_entry['holdout_typical_MAE'])

keep = {
    'objective', 'eval_metric', 'colsample_bytree', 'gamma', 'learning_rate',
    'max_bin', 'max_depth', 'min_child_weight', 'n_estimators',
    'reg_alpha', 'reg_lambda', 'subsample', 'tree_method', 'random_state', 'n_jobs',
}
xgb_params = {k: v for k, v in xgb_entry['params'].items() if k in keep and v is not None}
xgb_params

## 4. Обучение XGB и проверка качества

In [ ]:
model = xgb.XGBRegressor(**xgb_params)
model.fit(X_train_s, y_train)

pred_test = model.predict(X_test_s)
mae  = mean_absolute_error(y_test, pred_test)
rmse = mean_squared_error(y_test, pred_test) ** 0.5
mdae = median_absolute_error(y_test, pred_test)
print(f'holdout (typical, <960):  MAE={mae:.2f}  RMSE={rmse:.2f}  MdAE={mdae:.2f}')

## 5. SHAP: глобальная важность фичей

In [ ]:
explainer = shap.TreeExplainer(model)

bg_idx = np.random.RandomState(SEED).choice(len(X_train_s), size=min(1000, len(X_train_s)), replace=False)
shap_values_bg = explainer.shap_values(X_train_s[bg_idx])

shap.summary_plot(shap_values_bg, features=X_train_s[bg_idx], feature_names=feature_cols, show=True, max_display=15)

## 6. Утилиты: читаемые имена фичей и сборка вектора из параметров задачи

In [ ]:
_LABELS = {
    'Original Story Points': 'Story Points',
    'created_dow': 'День недели', 'created_month': 'Месяц',
    'created_year': 'Год', 'created_day': 'День месяца',
    'created_hour': 'Час создания',
    'team_prev_cnt': 'Завершённых задач у команды',
    'team_tasks_30d': 'Задач команды за 30 дней',
}

def human_name(col: str) -> str:
    if col in _LABELS:
        return _LABELS[col]
    for prefix, label in [('Приоритет_', 'Приоритет'), ('Тип_', 'Тип'), ('Команда_', 'Команда')]:
        if col.startswith(prefix):
            return f'{label}: {col[len(prefix):]}'
    return f'Тег: {col}'

team_cols = [c for c in feature_cols if c.startswith('Команда_')]
team_stats = {}
for tc in team_cols:
    name = tc.replace('Команда_', '')
    mask = train[tc] == 1
    if mask.any():
        team_stats[name] = {
            'team_prev_cnt': float(train.loc[mask, 'team_prev_cnt'].mean()),
            'team_tasks_30d': float(train.loc[mask, 'team_tasks_30d'].mean()),
        }

def build_feature_vector(story_points, priority, task_type, team, tags, created_at):
    row = {c: 0.0 for c in feature_cols}
    row['Original Story Points'] = float(story_points)
    row['created_dow']   = created_at.weekday()
    row['created_month'] = created_at.month
    row['created_year']  = created_at.year
    row['created_day']   = created_at.day
    row['created_hour']  = created_at.hour
    ts = team_stats.get(team, {'team_prev_cnt': 0.0, 'team_tasks_30d': 0.0})
    row['team_prev_cnt']  = ts['team_prev_cnt']
    row['team_tasks_30d'] = ts['team_tasks_30d']
    for prefix, val in [('Приоритет_', priority), ('Тип_', task_type), ('Команда_', team)]:
        col = f'{prefix}{val}'
        if col in row:
            row[col] = 1.0
    for tag in tags or []:
        if tag in row:
            row[tag] = 1.0
    raw = np.array([row[c] for c in feature_cols], dtype=np.float32).reshape(1, -1)
    scaled = scaler.transform(raw).astype(np.float32)
    np.nan_to_num(scaled, copy=False)
    return raw, scaled

## 7. Предсказание + локальный SHAP для одной задачи

In [ ]:
def predict_and_shap_scaled(scaled, raw, top_k=8):
    pred_hours = float(model.predict(scaled)[0])
    pred_hours = max(0.0, pred_hours)

    sv = explainer.shap_values(scaled).flatten()
    base_value = float(np.array(explainer.expected_value).flatten()[0])
    order = np.argsort(np.abs(sv))[::-1][:top_k]

    factors = [
        {
            'feature': feature_cols[i],
            'label':   human_name(feature_cols[i]),
            'value':   float(raw[0, i]),
            'shap_hours': round(float(sv[i]), 2),
            'direction': 'увеличивает' if sv[i] > 0 else 'уменьшает',
        }
        for i in order
    ]
    return {
        'predicted_hours': round(pred_hours, 1),
        'predicted_days':  round(pred_hours / 8.0, 1),
        'base_value':      round(base_value, 2),
        'top_factors':     factors,
        'shap_raw':        sv,
        'scaled':          scaled,
    }

# Берём реальный сэмпл из test — чтобы сравнить прогноз с actual
rng = np.random.RandomState(7)
sample_pos = int(rng.randint(0, len(test)))
sample_row = test.iloc[sample_pos]

actual_hours = float(sample_row[TARGET])
raw_sample = sample_row[feature_cols].values.astype(np.float32).reshape(1, -1)
scaled_sample = scaler.transform(raw_sample).astype(np.float32)
np.nan_to_num(scaled_sample, copy=False)

result = predict_and_shap_scaled(scaled_sample, raw_sample)

# Восстанавливаем человеко-читаемые метаданные задачи из one-hot
def _pick(prefix):
    cols = [c for c in feature_cols if c.startswith(prefix)]
    for c in cols:
        if float(sample_row[c]) == 1.0:
            return c[len(prefix):]
    return None

task_meta = {
    'story_points': float(sample_row['Original Story Points']),
    'priority':     _pick('Приоритет_'),
    'type':         _pick('Тип_'),
    'team':         _pick('Команда_'),
    'created_dow':  int(sample_row['created_dow']),
    'created_month':int(sample_row['created_month']),
    'created_year': int(sample_row['created_year']),
}

error = result['predicted_hours'] - actual_hours
print(f"Actual:   {actual_hours:.1f} ч ({actual_hours/8:.1f} д)")
print(f"Прогноз:  {result['predicted_hours']} ч ({result['predicted_days']} д)")
print(f"Ошибка:   {error:+.1f} ч")
print(f"Base value: {result['base_value']} ч")
print()
print('Параметры задачи:', task_meta)
print()
for f in result['top_factors']:
    print(f"  {f['label']:<40} {f['shap_hours']:+7.2f} ч  ({f['direction']})")

## 8. Локальный waterfall-plot (SHAP)

In [ ]:
expl = shap.Explanation(
    values=result['shap_raw'],
    base_values=result['base_value'],
    data=result['scaled'][0],
    feature_names=[human_name(c) for c in feature_cols],
)
shap.plots.waterfall(expl, max_display=12, show=True)

## 9. LLM-слой: ChatGPT объясняет прогноз и узкие места

Задаём API-ключ (или подхватываем из окружения) и шлём SHAP-контекст в модель.

In [ ]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')  # или впиши строкой
OPENAI_MODEL   = 'gpt-4o-mini'

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

def explain_with_llm(task_meta: dict, result: dict, language: str = 'ru') -> str:
    if client is None:
        raise RuntimeError('OPENAI_API_KEY не задан')

    factors_txt = '\n'.join(
        f"- {f['label']} (значение={f['value']:g}): SHAP={f['shap_hours']:+.2f} ч ({f['direction']} прогноз)"
        for f in result['top_factors']
    )

    sys_msg = (
        'Ты — ассистент-аналитик процессов разработки. Объясняешь прогнозы ML-модели '
        '(оттюненный XGBoost, регрессия длительности задачи в часах) через SHAP-значения. '
        'Твоя задача: (1) коротко объяснить, почему модель предсказала именно такое время, '
        '(2) выделить узкие места — факторы, которые сильнее всего увеличивают срок, '
        '(3) дать 2–3 практические рекомендации по их снижению. '
        f'Отвечай на {"русском" if language == "ru" else "английском"} языке, по делу, '
        'без воды. Не выдумывай фичи — опирайся только на переданные данные.'
    )

    user_msg = f"""Параметры задачи:
{json.dumps(task_meta, ensure_ascii=False, default=str, indent=2)}

Прогноз модели (оттюненный XGBoost):
- Часы: {result['predicted_hours']}
- Дней (по 8 ч): {result['predicted_days']}

SHAP base value (средний прогноз по трейну, ч): {result['base_value']}
Топ факторов влияния на этот прогноз:
{factors_txt}

Сформируй ответ в трёх секциях:
1) **Почему такой прогноз** — 2–4 предложения.
2) **Узкие места** — буллеты по факторам с положительным SHAP (увеличивают срок).
3) **Что сделать** — 2–3 конкретные рекомендации.
"""

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': user_msg},
        ],
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

## 10. Демо: прогноз + SHAP + LLM-объяснение

In [ ]:
print(f"Actual:   {actual_hours:.1f} ч ({actual_hours/8:.1f} д)")
print(f"Прогноз:  {result['predicted_hours']} ч ({result['predicted_days']} д)")
print(f"Ошибка:   {result['predicted_hours'] - actual_hours:+.1f} ч")
print()

if client is not None:
    explanation = explain_with_llm(task_meta, result)
    print(explanation)
else:
    print('OPENAI_API_KEY не задан — пропускаем вызов LLM. Задай ключ и перезапусти ячейку.')